In [ ]:
# Copyright (c) 2025, Meredith Lochhead
# All rights reserved.
#
# This source code is licensed under the BSD 3-Clause License found in the
# LICENSE file in the root directory of this source tree.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium 
import contextily as ctx
from matplotlib.patches import Patch
from shapely.geometry import box
import os
import sys
from matplotlib.ticker import ScalarFormatter, MaxNLocator
import seaborn as sns
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# Import functions for inventory generation 
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
fxn_dir = os.path.join(parent_dir, "inventory_generation_functions")
sys.path.append(fxn_dir)

import functions_general as fxns
import functions_preprocessing as pre 
import functions_disagreement_and_gaps as resolve

In [ ]:
# Figure Directory 
fig_dir = './Figures/Results/'
os.makedirs(fig_dir, exist_ok=True)

# Specify resolution of images 
dpi = 500

In [ ]:
# Set plotting CRS values for data manipulation and plotting
crs_main = '26910' # Used for data manipulation and storage
crs_plot = '4269' # Used for plotting 

# HAYWARD BOUNDS
xbounds = [-122.15, -122.02]
ybounds = [37.60, 37.69]

# Define the color palette
stanford_palette = [
    "#034185", 
    "#a14059",  
    "#277b8eff",  
    "#ace1fbff",    
    "#a7a7d6ff",
    "red"]

# Set the palette in seaborn
sns.set_palette(sns.color_palette(stanford_palette))
colors = stanford_palette

## FEATURE COMPARISONS

In [ ]:
# Define inventories
point = fxns.json_to_gdf('./R2D_Analysis/Inventories/NSI_Point/R2D_Inventory.json', crs_main)
spatial = fxns.json_to_gdf('./R2D_Analysis/Inventories/NSI_SpatialJoin/R2D_Inventory.json', crs_main)
national = fxns.json_to_gdf('./R2D_Analysis/Inventories/Synthesized_National/R2D_Inventory.json', crs_main)
local = fxns.json_to_gdf('./R2D_Analysis/Inventories/Synthesized_Local/R2D_Inventory.json', crs_main)
best = fxns.json_to_gdf('./R2D_Analysis/Inventories/Best_Estimate/R2D_Inventory.json', crs_main)

# Footprints
footprints = fxns.json_to_gdf('./Input_Data/ProcessedFootprints/Hayward_Footprints.json', crs_main)


In [ ]:
inventories = [point, spatial, national, local, best]
inventory_names = ['NSI Points', 'NSI Spatial Join', 'Synthesized National',  'Synthesized Local', 'Best Estimate']

# Set inventory lengths
inventory_lengths = [len(gdf) for gdf in inventories]

# Set population values 
local['NightPopulation'] = 0
pop_values = [gdf['NightPopulation'].sum() for gdf in inventories]

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 5), dpi=dpi)

# --- Data ---
x = np.arange(len(inventory_names))  # shared x-axis positions
width = 0.35

# --- First axis (bar set 1) ---
bars1 = ax1.bar(
    x - width/2,
    [v / 1000 for v in inventory_lengths],
    width=width,
    color=colors[:5],
    alpha=1,
    label="Total Buildings")

ax1.set_ylabel("Total Number of Buildings (thousands)")
ax1.set_xticks(x)
ax1.set_xticklabels(inventory_names, rotation=30, ha='center')

# --- Second axis (bar set 2) ---
ax2 = ax1.twinx()

bars2 = ax2.bar(
    x + width/2,
    [v / 1000 for v in pop_values],
    width=width,
    color=colors,
    hatch='xx',  
    alpha=0.8,
    label="Nighttime Population")

ax2.set_ylabel("Total Nighttime Population (thousands)")

# --- Title ---
plt.title("Total Number of Buildings and Nighttime Population by Inventory")


ax1.set_ylim(0, 45) 
ax2.set_ylim(0, 194)


# --- Legend ---
solid_patch = Patch(facecolor='lightgray', edgecolor='black', label='Number of Buildings')
hatch_patch = Patch(facecolor='lightgray', edgecolor='black', hatch='xx', label='Nighttime Population')
ax1.legend(handles=[solid_patch, hatch_patch], loc='upper right')

plt.tight_layout()

plt.savefig(fig_dir + "total_count_and_pop_combined.svg", format="svg",
            bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "total_count_and_pop_combined.png",
            bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "total_count_and_pop_combined.pdf",
            bbox_inches='tight', pad_inches=0)

# plt.close()

### OCCUPANCY CLASS

In [ ]:
# inventories = [point, spatial, national, local, best]
# inventory_names = ['NSI Points', 'NSI Spatial Join', 'Synthesized National', 'Synthesized Local', 'Best Estimate']
# inventory_lengths = [len(gdf) for gdf in inventories]

In [ ]:
# Function to map categories
def simplify_occ(value):
    if 'RES' in value:
        return 'RES'
    elif 'IND' in value:
        return 'IND'
    elif 'COM' in value:
        return 'COM'
    elif 'GOV' in value:
        return 'GOV'
    elif 'REL' in value:
        return 'REL'
    elif 'EDU' in value:
        return 'EDU'
    elif 'AGR' in value:
        return 'AGR'
    else:
        return 'OTHER'

def remove_mixed(value):
    # if value == 'RES1M':
    if 'RES1' in value:
        return 'RES1'
    elif 'RES3AM' in value:
        return 'RES3A'
    elif 'RES3BM' in value:
        return 'RES3B'
    elif 'RES3CM' in value:
        return 'RES3C'
    elif 'RES3DM' in value:
        return 'RES3D'
    elif 'RES3EM' in value:
        return 'RES3E'
    elif 'RES3FM' in value:
        return 'RES3F'
    else:
        return value

    
# Create columns with overarching categories
for inventory in inventories: 
    inventory['OccupancyClass_General'] = inventory['OccupancyClass_Actual'].apply(simplify_occ)


# Create columns with overarching categories
for inventory in inventories: 
    inventory['OccupancyClass'] = inventory['OccupancyClass_Actual'].apply(remove_mixed)

In [ ]:
### PLOT COUNTS OF GENERAL OCCUPANCY CLASS CATEGORIES
occupancy_classes = ['RES', 'IND', 'COM', 'GOV', 'EDU']

# Count occurrences for each occupancy class in each inventory
occ_counts = {
    name: [inventory['OccupancyClass_General'].str.contains(occ, case=False, na=False).sum() for occ in occupancy_classes]
    for name, inventory in zip(inventory_names, inventories)}

# Create a DataFrame for visualization
counts_df = pd.DataFrame(occ_counts, index=occupancy_classes)

# Create DataFrame using total building counts 
len_df = pd.DataFrame([inventory_lengths], columns=inventory_names)

# Combine DataFrames
df = counts_df.copy() #pd.concat([len_df, counts_df])
# df.index = ['All Buildings', 'Residential', 'Industrial', 'Commerical', 'Governmental', 'Educational']
df.index = ['Residential', 'Industrial', 'Commerical', 'Governmental', 'Educational']

# Plotting
plt.figure(dpi = dpi)
ax = df.plot(kind='bar', figsize=(6, 4), width=0.7, color = colors)
# ax.set_ylim([30000,42000])
plt.xticks(rotation=45) 
plt.ylabel('Number of Buildings')
ax.set_title('Occupancy Class Cateogry Counts Across Inventories')
plt.savefig(fig_dir + "oc_category_count_NOBOUNDS.svg", format="svg",  bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "oc_category_count_NOBOUNDS.png",  bbox_inches='tight')
# plt.close()

In [ ]:
### PLOT COUNTS OF GENERAL OCCUPANCY CLASS CATEGORIES
occupancy_classes = ['GOV', 'EDU']

# Count occurrences for each occupancy class in each inventory
occ_counts = {
    name: [inventory['OccupancyClass_General'].str.contains(occ, case=False, na=False).sum() for occ in occupancy_classes]
    for name, inventory in zip(inventory_names, inventories)}

# Create a DataFrame for visualization
counts_df = pd.DataFrame(occ_counts, index=occupancy_classes)

# Create DataFrame
counts_df.index = ['Governmental', 'Educational']

# Plotting
plt.figure(dpi = dpi)
ax = counts_df.plot(kind='bar', figsize=(2,2.5), width=0.7, color = colors)
plt.xticks(rotation=45) 
plt.ylabel('Number of Buildings')
ax.legend().set_visible(False)
plt.savefig(fig_dir + "oc_category_count_EDUGOV.svg", format="svg",  bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "oc_category_count_EDUGOV.png",  bbox_inches='tight')
# plt.close()


In [ ]:
### PLOT COUNTS OF SPECIFIC RESIDENTIAL OCCUPANCY CLASS CATEGORIES

occupancy_classes = ['RES1','RES2','RES3A','RES3B','RES3C','RES3D','RES3E','RES3F']


# Count occurrences for each occupancy class in each inventory
occ_counts = {
    name: [inventory['OccupancyClass'].str.contains(occ, case=False, na=False).sum() for occ in occupancy_classes]
    for name, inventory in zip(inventory_names, inventories)}

# Create a DataFrame for visualization
counts_df = pd.DataFrame(occ_counts, index=occupancy_classes)

# Plotting
plt.figure()
ax = counts_df.plot(kind='bar', figsize=(6, 4.5), width=0.7, color = colors)
ax.set_ylim([23500,28500])
ax.set_yticks(np.arange(23500, 28501, 500))
ax.set_title('Residential Occupancy Class Counts Across Inventories')
plt.savefig(fig_dir + "oc_res_count_top.svg", format="svg",  bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "oc_res_count_top.png", bbox_inches='tight', pad_inches=0, dpi=dpi)
# plt.close()

plt.figure()
ax = counts_df.plot(kind='bar', figsize=(6, 3), width=0.7, color = colors)
ax.set_ylim([0,3800])
ax.legend().set_visible(False)
plt.savefig(fig_dir + "oc_res_count_bot.svg", format="svg",  bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "oc_res_count_bot.png", bbox_inches='tight', pad_inches=0, dpi=dpi)
# plt.close()

#### EXAMINE OCCUPANCY CLASS FOR DIFFERENT INVENTORIES 

In [ ]:
def get_agree(feature, source1, source2):
    # Merge gdfs on footprint_id
    merged = source1[['FootprintID', feature]].merge(source2[['FootprintID',feature]], on='FootprintID', suffixes=('_source1', '_source2'))

    # Bring in footprint geometry and convert to gdf
    merged_footprints = merged.merge(footprints[['FootprintID','geometry']], on='FootprintID')
    merged_footprints = gpd.GeoDataFrame(merged_footprints, geometry='geometry')
    merged_footprints.set_crs(crs_main, inplace=True)

    # Separate agree and disagree
    agree = merged_footprints[merged_footprints[f'{feature}_source1'] == merged_footprints[f'{feature}_source2']]
    disagree = merged_footprints[merged_footprints[f'{feature}_source1'] != merged_footprints[f'{feature}_source2']]

    agree = agree.to_crs(crs_plot)
    disagree = disagree.to_crs(crs_plot)

    return agree, disagree

In [ ]:
## OCCUPANCY CLASS GENERAL
feature = 'OccupancyClass_General'

# Find for spatial join and best inventory 
agree, disagree = get_agree(feature, local, spatial)
print(len(agree) / (len(agree) + len(disagree)), 'match between spatial join and local')

# Find for national and best inventory 
agree, disagree = get_agree(feature, local, national)
print(len(agree) / (len(agree) + len(disagree)), 'match between national best and local\n')

## OCCUPANCY CLASS
feature = 'OccupancyClass'

# Find for spatial join and best inventory 
agree, disagree = get_agree(feature, local, spatial)
print(len(agree) / (len(agree) + len(disagree)), 'match between spatial join and local')

# Find for national and best inventory 
agree, disagree = get_agree(feature, local, national)
print(len(agree) / (len(agree) + len(disagree)), 'match between national best and local\n')

In [ ]:
### COMPARE OCCUPANCY CLASS (GENERAL) AND OCCUPANCY CLASS (SPECIFIC) FOR SPATIAL JOIN
fig, ax = plt.subplots(1,2, figsize = (9,9), sharex = True, sharey = True, dpi = dpi)

# Find for spatial join local
agree, disagree = get_agree('OccupancyClass_General', local, spatial)
print(len(agree) / (len(agree) + len(disagree)), 'match between spatial join and local')
disagree.plot(ax=ax[0], color = 'tab:red')
agree.plot(ax=ax[0], color = 'tab:blue')
ax[0].set_title('Occupancy Class Category', fontsize = 12)
agree0 = round(len(agree) / (len(agree) + len(disagree))*100)

# Find for spatial join and local
agree, disagree = get_agree('OccupancyClass', local, spatial)
print(len(agree) / (len(agree) + len(disagree)), 'match between spatial join and local')
agree1 = round(len(agree) / (len(agree) + len(disagree))*100)
disagree.plot(ax=ax[1], color = 'tab:red')
agree.plot(ax=ax[1], color = 'tab:blue')
ax[1].set_title('Specific Occupancy Class', fontsize = 12)


# Add bounds
ax[0].set_xlim(xbounds)
ax[0].set_ylim(ybounds)

# Add the OpenStreetMap basemap
ctx.add_basemap(ax[0], crs='EPSG:4326', source=ctx.providers.CartoDB.PositronNoLabels, zoom = 12)
ctx.add_basemap(ax[1], crs='EPSG:4326', source=ctx.providers.CartoDB.PositronNoLabels, zoom = 12)

# Add box to remove distortion
bb1 = box(-122.091127,37.636333,-122.080399,37.643282)
gdf_bb = gpd.GeoDataFrame({"geometry": [bb1]}, crs=crs_plot)
gdf_bb.boundary.plot(ax=ax[0], alpha = 0)
gdf_bb.boundary.plot(ax=ax[1], alpha = 0)

# Set tick marks for x axis 
ax[0].xaxis.set_major_locator(MaxNLocator(nbins=5))

# Add a legend
legend_patches0 = [
    mpatches.Patch(color='tab:blue', label=f'Agrees ({agree0}%)'),
    mpatches.Patch(color='tab:red', label=f'Disagrees ({100 - agree0}%)')]
legend_patches1 = [
    mpatches.Patch(color='tab:blue', label=f'Agrees ({agree1}%)'),
    mpatches.Patch(color='tab:red', label=f'Disagrees ({100 - agree1}%)')]

ax[0].legend(handles=legend_patches0, loc='upper left', fontsize = 10)
ax[1].legend(handles=legend_patches1, loc='upper left', fontsize = 10)

plt.tight_layout()
plt.savefig(fig_dir + "oc_category_vs_specific.svg", format="svg",  bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "oc_category_vs_specific.png",  bbox_inches='tight')
# plt.close()

In [ ]:
### COMPARE OCCUPANCY CLASS (SPECIFIC) FOR SPATIAL JOIN AND NATIONAL BEST VS LOCALLY DEVELOPED -- RESIDENTIAL ONLY 

feature = 'OccupancyClass'

best_res = best[best['OccupancyClass'].str.contains('RES')]
spatial_res = spatial[spatial['OccupancyClass'].str.contains('RES')]
national_res = national[national['OccupancyClass'].str.contains('RES')]
local_res = local[local['OccupancyClass'].str.contains('RES')]

fig, ax = plt.subplots(1,2, figsize = (12,12), sharex = True, sharey = True, dpi = dpi)

# Find for spatial join and best inventory 
agree, disagree = get_agree(feature, local_res, spatial_res)
print(len(agree) / (len(agree) + len(disagree)), 'match between spatial join and local')
disagree.plot(ax=ax[0], color = 'tab:red')
agree.plot(ax=ax[0], color = 'tab:blue')
ax[0].set_title('NSI Spatial Join vs Synthesized Local \n Residential Occupancy Class')
agree0 = round(len(agree) / (len(agree) + len(disagree))*100)

# Find for national and best inventory 
agree, disagree = get_agree(feature, local_res, national_res)
print(len(agree) / (len(agree) + len(disagree)), 'match between national best and local')
disagree.plot(ax=ax[1], color = 'tab:red')
agree.plot(ax=ax[1], color = 'tab:blue')
ax[1].set_title('Synthesized National vs Synthesized Local \n Residential Occupancy Class')
agree1 = round(len(agree) / (len(agree) + len(disagree))*100)

ax[0].set_xlim([-122.122, -122.045])
ax[0].set_ylim([37.615, 37.67])

# Add the OpenStreetMap basemap
ctx.add_basemap(ax[0], crs='EPSG:4326', source=ctx.providers.CartoDB.PositronNoLabels, zoom = 14)
ctx.add_basemap(ax[1], crs='EPSG:4326', source=ctx.providers.CartoDB.PositronNoLabels, zoom = 14)

# Add box to remove distortion
bb1 = box(-122.091127,37.636333,-122.080399,37.643282)
gdf_bb = gpd.GeoDataFrame({"geometry": [bb1]}, crs=crs_plot)
gdf_bb.boundary.plot(ax=ax[0], alpha = 0)
gdf_bb.boundary.plot(ax=ax[1], alpha = 0)

# Set tick marks for x axis 
ax[0].xaxis.set_major_locator(MaxNLocator(nbins=5))

# Add a legend
legend_patches0 = [
    mpatches.Patch(color='tab:blue', label=f'Agrees ({agree0}%)'),
    mpatches.Patch(color='tab:red', label=f'Disagrees ({100 - agree0}%)')]
legend_patches1 = [
    mpatches.Patch(color='tab:blue', label=f'Agrees ({agree1}%)'),
    mpatches.Patch(color='tab:red', label=f'Disagrees ({100 - agree1}%)')]


ax[0].legend(handles=legend_patches0, loc='upper right', fontsize = 10)
ax[1].legend(handles=legend_patches1, loc='upper right', fontsize = 10)

plt.savefig(fig_dir + "oc_spatial_vs_synthesized_res.svg", format="svg",  bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "oc_spatial_vs_synthesized_res.png",  bbox_inches='tight', pad_inches=0)
# plt.close()

In [ ]:
# CONFUSION MATRICIES FOR DISAGREEMENTS
spatial_res = spatial[spatial['OccupancyClass'].str.contains('RES1|RES2|RES3')]
national_res = national[national['OccupancyClass'].str.contains('RES1|RES2|RES3')]
local_res = local[local['OccupancyClass'].str.contains('RES1|RES2|RES3')]

# Local and Spatial Join
agree, disagree = get_agree(feature, local_res, spatial_res)
conf_matrix = pd.crosstab(disagree['OccupancyClass_source1'], disagree['OccupancyClass_source2'], rownames=['Synthesized Local Occupancy Class'], colnames=['NSI Spatial Join Occupancy Class'])
fig, ax = plt.subplots(1,1, figsize = (6,5), dpi = dpi)
sns.heatmap(conf_matrix, ax=ax, annot=True, fmt='d', cmap='Blues')
plt.title("Disagreement in Residential Occupancy Class")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0, va='center')
plt.savefig(fig_dir + "oc_confusion_spatial.svg", format="svg",  bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "oc_confusion_spatial.png", bbox_inches='tight', pad_inches=0)
# plt.close()

# Local and National
agree, disagree = get_agree(feature, local_res, national_res)
conf_matrix = pd.crosstab(disagree['OccupancyClass_source1'], disagree['OccupancyClass_source2'], rownames=['Synthesized Local Occupancy Class'], colnames=['Synthesized National Occupancy Class'])
fig, ax = plt.subplots(1,1, figsize = (6,5), dpi = dpi)
sns.heatmap(conf_matrix, ax=ax, annot=True, fmt='d', cmap='Blues')
plt.title("Disagreement in Residential Occupancy Class")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0, va='center')
plt.savefig(fig_dir + "oc_confusion_national.svg", format="svg",  bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "oc_confusion_national.png", bbox_inches='tight', pad_inches=0)
# plt.close()

In [ ]:
# # # ### UNCOMMENT CODE TO PLOT INTERACTIVE MAP WITH FOOTPRINTS AND POINTS
# # Load building footprints 
# footprints = fxns.json_to_gdf('./Input_Data/ProcessedFootprints/Hayward_Footprints.json', crs_main)


# # Create a base map
# m = folium.Map(location=[footprints.copy().to_crs(crs_plot).geometry.iloc[0].centroid.y, footprints.copy().to_crs(crs_plot).geometry.iloc[0].centroid.x], zoom_start=12)


# # # Add footprints (polygons)
# folium.GeoJson(footprints.copy().to_crs(crs_plot), color = 'gray').add_to(m)
# folium.GeoJson(national.copy().to_crs(crs_plot), color = 'green').add_to(m)
# folium.GeoJson(nat_res1_loc_res3c.copy().to_crs(crs_plot), color = 'red').add_to(m)

# # # Add remaining points     
# # for idx, row in nat_res1_loc_res3c.copy().iterrows():
# #     folium.CircleMarker(location=[row.geometry.y, row.geometry.x], 
# #                         radius=1, 
# #                         color='red', 
# #                         fill=True, 
# #                         fill_color='red').add_to(m)
# # # Add remaining points     
# # for idx, row in local2.copy().iterrows():
# #     folium.CircleMarker(location=[row.geometry.y, row.geometry.x], 
# #                         radius=1, 
# #                         color='blue', 
# #                         fill=True, 
#                         # fill_color='blue').add_to(m)
# display(m)

In [ ]:
# Occupancy Class, All Footprints Excluding RES1

feature = 'OccupancyClass'

best_nonres1 = best[best['OccupancyClass'] != 'RES1']
spatial_res = spatial[spatial['OccupancyClass'] != 'RES1']
national_res = national[national['OccupancyClass'] != 'RES1']
local_res = local[local['OccupancyClass'] != 'RES1']

# Find for spatial join and best inventory 
agree, disagree = get_agree(feature, local_res, spatial_res)
print(len(agree) / (len(agree) + len(disagree)), 'match between spatial join and local')

# Find for national and best inventory 
agree, disagree = get_agree(feature, local_res, national_res)
print(len(agree) / (len(agree) + len(disagree)), 'match between national best and local')




# Occupancy Class, All Residential Footprints Excluding RES1

feature = 'OccupancyClass'

best_nonres1 = best[(best['OccupancyClass'] != 'RES1') & (best['OccupancyClass'].str.contains('RES'))]
spatial_res = spatial[(spatial['OccupancyClass'] != 'RES1') & (spatial['OccupancyClass'].str.contains('RES'))]
national_res = national[(national['OccupancyClass'] != 'RES1') & (national['OccupancyClass'].str.contains('RES'))]
local_res = local[(local['OccupancyClass'] != 'RES1') & (local['OccupancyClass'].str.contains('RES'))]

# Find for spatial join and best inventory 
agree, disagree = get_agree(feature, local_res, spatial_res)
print(len(agree) / (len(agree) + len(disagree)), 'match between spatial join and local')

# Find for national and best inventory 
agree, disagree = get_agree(feature, local_res, national_res)
print(len(agree) / (len(agree) + len(disagree)), 'match between national best and local')

### BUILDING TYPE

In [ ]:
### PLOT COUNTS OF BUILDING TYPE CATEGORIES

bldg_types = ['W', 'H', 'M', 'C', 'S']

# Count occurrences for each occupancy class in each inventory
occ_counts = {
    name: [inventory['BuildingType'].str.contains(occ, case=False, na=False).sum() for occ in bldg_types]
    for name, inventory in zip(inventory_names, inventories)}

# Create a DataFrame for visualization
counts_df = pd.DataFrame(occ_counts, index=bldg_types)

# Plotting
plt.figure()
ax = counts_df.plot(kind='bar', figsize=(5, 5), width=0.7, color = colors)
ax.set_title('Building Type Counts Across Inventories')

plt.xticks(rotation=0, ha='right')
plt.savefig(fig_dir + "bldg_type_counts.svg", format="svg",  bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "bldg_type_counts.png", bbox_inches='tight', pad_inches=0, dpi = dpi)
# plt.close()


In [ ]:
bb1 = box(-122.114039,37.622866,-122.099277,37.631549)

feature = 'BuildingType'

### SPATIAL JOIN AND LOCAL
fig, ax = plt.subplots(1, 1, figsize=(5,5), sharex = True, sharey=True, dpi = dpi)
agree, disagree = get_agree(feature, local, spatial)
bb_coords = list(bb1.exterior.coords)
trim_agree = agree[agree.geometry.intersects(bb1)]
trim_disagree = disagree[disagree.geometry.intersects(bb1)]

trim_agree.plot(ax=ax,color = 'tab:blue')
trim_disagree.plot(ax=ax,color = 'tab:red')
ctx.add_basemap(ax, crs='EPSG:4326', source=ctx.providers.CartoDB.PositronNoLabels, zoom=16)
gdf_bb = gpd.GeoDataFrame({"geometry": [bb1]}, crs=crs_plot)
gdf_bb.boundary.plot(ax=ax, edgecolor="red", linewidth=2, alpha = 0)
plt.xlim([bb_coords[2][0], bb_coords[0][0]])
plt.ylim([bb_coords[0][1], bb_coords[2][1]])
ax.xaxis.set_major_formatter(ScalarFormatter(useOffset=False))
ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))
ax.xaxis.set_major_locator(MaxNLocator(nbins=7))  # Adjust `nbins` as needed for fewer ticks
ax.set_xticks([])  # Remove x-axis ticks
ax.set_yticks([])  # Remove y-axis ticks
ax.set_xticklabels([])  # Remove x-axis labels
ax.set_yticklabels([])  # Remove y-axis labels
plt.savefig(fig_dir + "bldg_type_cutout1.svg", format="svg",  bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "bldg_type_cutout1.png",  bbox_inches='tight', pad_inches=0)
# plt.close()

### NATIONAL AND LOCAL
fig, ax = plt.subplots(1, 1, figsize=(5,5), sharex = True, sharey=True, dpi = dpi)
agree, disagree = get_agree(feature, local, national)
bb_coords = list(bb1.exterior.coords)
trim_agree = agree[agree.geometry.intersects(bb1)]
trim_disagree = disagree[disagree.geometry.intersects(bb1)]

trim_agree.plot(ax=ax,color = 'tab:blue')
trim_disagree.plot(ax=ax,color = 'tab:red')
ctx.add_basemap(ax, crs='EPSG:4326', source=ctx.providers.CartoDB.PositronNoLabels, zoom=16)
gdf_bb = gpd.GeoDataFrame({"geometry": [bb1]}, crs=crs_plot)
gdf_bb.boundary.plot(ax=ax, edgecolor="red", linewidth=2, alpha = 0)
plt.xlim([bb_coords[2][0], bb_coords[0][0]])
plt.ylim([bb_coords[0][1], bb_coords[2][1]])
ax.xaxis.set_major_formatter(ScalarFormatter(useOffset=False))
ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))
ax.xaxis.set_major_locator(MaxNLocator(nbins=7))  # Adjust `nbins` as needed for fewer ticks

ax.set_xticks([])  # Remove x-axis ticks
ax.set_yticks([])  # Remove y-axis ticks
ax.set_xticklabels([])  # Remove x-axis labels
ax.set_yticklabels([])  # Remove y-axis labels
plt.savefig(fig_dir + "bldg_type_cutout2.svg", format="svg",  bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "bldg_type_cutout2.png",  bbox_inches='tight', pad_inches=0)
# plt.close()

In [ ]:
### COMPARE OCCUPANCY CLASS (GENERAL) FOR SPATIAL JOIN AND NATIONAL BEST VS LOCALLY DEVELOPED 

feature = 'BuildingType'

fig, ax = plt.subplots(1,2, figsize = (12,12), sharex = True, sharey = True, dpi = dpi)

# Find for spatial join and best inventory 
agree, disagree = get_agree(feature, local, spatial)
agree0 = round(len(agree) / (len(agree) + len(disagree))*100)
print(len(agree) / (len(agree) + len(disagree))*100, 'match between spatial join and local')
disagree.plot(ax=ax[0], color = 'tab:red')
agree.plot(ax=ax[0], color = 'tab:blue')
ax[0].set_title('NSI Spatial Join vs Synthesized Local\n Building Type')

# Find for national and best inventory 
agree, disagree = get_agree(feature, local, national)
agree1 = round(len(agree) / (len(agree) + len(disagree))*100)
print(len(agree) / (len(agree) + len(disagree))*100, 'match between national best and local')
disagree.plot(ax=ax[1], color = 'tab:red')
agree.plot(ax=ax[1], color = 'tab:blue')
ax[1].set_title('Synthesized National vs Synthesized Local\n Building Type')


# Add bounds
ax[0].set_xlim(xbounds)
ax[0].set_ylim(ybounds)

# Add the OpenStreetMap basemap
ctx.add_basemap(ax[0], crs='EPSG:4326', source=ctx.providers.CartoDB.PositronNoLabels, zoom = 12)
ctx.add_basemap(ax[1], crs='EPSG:4326', source=ctx.providers.CartoDB.PositronNoLabels, zoom = 12)

# Add box to remove distortion
gdf_bb = gpd.GeoDataFrame({"geometry": [bb1]}, crs=crs_plot)
gdf_bb.boundary.plot(ax=ax[0], alpha = 1, color = 'black')
gdf_bb.boundary.plot(ax=ax[1], alpha = 1, color = 'black')

# Set tick marks for x axis 
ax[0].xaxis.set_major_locator(MaxNLocator(nbins=5))

# # Add a legend
# legend_patches = [
#     mpatches.Patch(color='tab:blue', label='Agrees'),
#     mpatches.Patch(color='tab:red', label='Disagrees'),
# ]

legend_patches0 = [
    mpatches.Patch(color='tab:blue', label=f'Agrees ({agree0}%)'),
    mpatches.Patch(color='tab:red', label=f'Disagrees ({100 - agree0}%)')]

legend_patches1 = [
    mpatches.Patch(color='tab:blue', label=f'Agrees ({agree1}%)'),
    mpatches.Patch(color='tab:red', label=f'Disagrees ({100 - agree1}%)')]


ax[0].legend(handles=legend_patches0, loc='upper right', fontsize = 10)
ax[1].legend(handles=legend_patches1, loc='upper right', fontsize = 10)

plt.savefig(fig_dir + "bldg_type_overview.svg", format="svg",  bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "bldg_type_overview.png",  bbox_inches='tight', pad_inches=0)
# plt.close()

In [ ]:
feature = 'StructureType'


# Find for spatial join and best inventory 
agree, disagree = get_agree(feature, local, spatial)
agree0 = round(len(agree) / (len(agree) + len(disagree))*100)
print(len(agree) / (len(agree) + len(disagree))*100, 'match between spatial join and local')
disagree.plot(ax=ax[0], color = 'tab:red')
agree.plot(ax=ax[0], color = 'tab:blue')
ax[0].set_title('NSI Spatial Join vs Synthesized Local\n Building Type')

# Find for national and best inventory 
agree, disagree = get_agree(feature, local, national)
agree1 = round(len(agree) / (len(agree) + len(disagree))*100)
print(len(agree) / (len(agree) + len(disagree))*100, 'match between national best and local')
disagree.plot(ax=ax[1], color = 'tab:red')
agree.plot(ax=ax[1], color = 'tab:blue')
ax[1].set_title('Synthesized National vs Synthesized Local\n Building Type')

In [ ]:
# Occupancy Class, All Footprints Excluding RES1

feature = 'StructureType'

best_nonres1 = best[best['OccupancyClass'] != 'RES1']
spatial_res = spatial[spatial['OccupancyClass'] != 'RES1']
national_res = national[national['OccupancyClass'] != 'RES1']
local_res = local[local['OccupancyClass'] != 'RES1']

# Find for spatial join and best inventory 
agree, disagree = get_agree(feature, local_res, spatial_res)
print(len(agree) / (len(agree) + len(disagree)), 'match between spatial join and local')

# Find for national and best inventory 
agree, disagree = get_agree(feature, local_res, national_res)
print(len(agree) / (len(agree) + len(disagree)), 'match between national best and local')




# Occupancy Class, All Residential Footprints Excluding RES1

feature = 'StructureType'

best_nonres1 = best[(best['OccupancyClass'] != 'RES1') & (best['OccupancyClass'].str.contains('RES'))]
spatial_res = spatial[(spatial['OccupancyClass'] != 'RES1') & (spatial['OccupancyClass'].str.contains('RES'))]
national_res = national[(national['OccupancyClass'] != 'RES1') & (national['OccupancyClass'].str.contains('RES'))]
local_res = local[(local['OccupancyClass'] != 'RES1') & (local['OccupancyClass'].str.contains('RES'))]

# Find for spatial join and best inventory 
agree, disagree = get_agree(feature, local_res, spatial_res)
print(len(agree) / (len(agree) + len(disagree)), 'match between spatial join and local')

# Find for national and best inventory 
agree, disagree = get_agree(feature, local_res, national_res)
print(len(agree) / (len(agree) + len(disagree)), 'match between national best and local')

## EVALUATE CENSUS UNIT ESTIMATION

In [ ]:
def assign_occ_class(value):

    if value == 1:
        occ = "RES1"
    elif value == 2: 
        occ = "RES3A"
    elif (value >= 3) & (value <= 4):
        occ = "RES3B"
    elif (value > 4) & (value < 10):
        occ = "RES3C"
    elif (value >= 10) & (value < 20):
        occ = "RES3D"
    elif (value >= 20) & (value <= 50):
        occ = "RES3E"
    elif (value > 50):
        occ = "RES3F"
    else: 
        print(value)
        occ = 'ERROR'
    
    return occ



def modulate_occ(s):
    if pd.isna(s) or s == '':
        return s

    # Address cases where details have been added in Occupancy Type
    if "GOV2" in s: 
        return "GOV2"
    elif "EDU1" in s: 
        return "EDU1"
    
    # Handle mixed use cases
    elif s[-1]=='M':  
        
        return s[:-1]  # Remove the last character
    return s  # Return the string unchanged if it ends with a number

In [ ]:
# Load national inventory 
national_all = fxns.json_to_gdf('./Inventory_Outputs/Synthesized_National/InventoryGeneration/Inventory_AllFields.json', crs_main)
best_all = fxns.json_to_gdf('./Inventory_Outputs/Best_Estimate/InventoryGeneration/Inventory_AllFields.json', crs_main)

national = national_all[['OccupancyClass_Best','NSI_MinResUnits','NSI_MaxResUnits','Units_CensusEstimate','Units_Best','Units_Best_Source',
          'FootprintID','National_Flag', 'geometry']]

best = best_all[['OccupancyClass_Best','Units_Best', 'Units_Best_Source','FootprintID','Local_Flag', 'National_Flag','geometry']]

# Separate out residential 
national_res = national[national['OccupancyClass_Best'].str.contains('RES1|RES3')]
best_res = best[best['OccupancyClass_Best'].str.contains('RES1|RES3')]

# Rename columns for merge
national_res = national_res.rename(columns={'Units_Best':'Units_CensusBest',
                                    'OccupancyClass_Best':'OccupancyClass_Best_National'})

In [ ]:
### COMPUTE OCCUPANCY CLASS BASED ON NUMBER OF UNITS

# Compute other methods for estimating units
national_res['Average'] = (national_res['NSI_MinResUnits'] + national_res['NSI_MaxResUnits'])/2

# Assign occupancy classes based on number units 
national_res.loc[:,'Units_CensusEstimate_OC'] = national_res['Units_CensusBest'].apply(assign_occ_class)
national_res.loc[:,'Average_OC'] = national_res['Average'].apply(assign_occ_class)
national_res.loc[:,'NSI_MinResUnits_OC'] = national_res['NSI_MinResUnits'].apply(assign_occ_class)
national_res.loc[:,'NSI_MaxResUnits_OC'] = national_res['NSI_MaxResUnits'].apply(assign_occ_class)


### REPEAT FOR COMBINED DATA TO COMPUTE CORRELATIONS
# Combine data
combo = national_res.merge(best_res, on = 'FootprintID')

# Compute other methods for estimating units
combo['Average'] = (combo['NSI_MinResUnits'] + combo['NSI_MaxResUnits'])/2

# Assign occupancy classes based on number units 
combo.loc[:,'Units_CensusEstimate_OC'] = combo['Units_CensusBest'].apply(assign_occ_class) ## MATCHES CLOSELY WITH OccupancyClass_Best_National
combo.loc[:,'Average_OC'] = combo['Average'].apply(assign_occ_class)
combo.loc[:,'NSI_MinResUnits_OC'] = combo['NSI_MinResUnits'].apply(assign_occ_class)
combo.loc[:,'NSI_MaxResUnits_OC'] = combo['NSI_MaxResUnits'].apply(assign_occ_class)

# Remove cases with no data
combo = combo[(combo['National_Flag_x']!=0) & (combo['Local_Flag']!=0)]


In [ ]:
# Look at correlation for above 50 units
cut = combo[combo['Units_Best']>50]
print(np.corrcoef(cut['Units_Best'], cut['Units_CensusBest'])[0,1])
print(np.corrcoef(cut['Units_Best'], cut['Average'])[0,1])
print(np.corrcoef(cut['Units_Best'], cut['NSI_MinResUnits'])[0,1])
print(np.corrcoef(cut['Units_Best'], cut['NSI_MaxResUnits'])[0,1])

# Look at correlation without RES1 data
cut = combo[combo['OccupancyClass_Best']!='RES1']
print('\n',np.corrcoef(cut['Units_Best'], cut['Units_CensusBest'])[0,1])
print(np.corrcoef(cut['Units_Best'], cut['Average'])[0,1])
print(np.corrcoef(cut['Units_Best'], cut['NSI_MinResUnits'])[0,1])
print(np.corrcoef(cut['Units_Best'], cut['NSI_MaxResUnits'])[0,1])

In [ ]:
fig,ax = plt.subplots(1,4, figsize = (12,4), sharex = True, sharey = True, dpi = dpi)

for i in range(4):
    x = np.linspace(0.5, 100, 1000)
    ax[i].plot(x, x, label='y = x', color='gray', alpha =0.2, linestyle = '-')
    ax[i].set_xlabel('Best Estimate Units')
ax[0].set_ylabel('Estimated Units')

alpha_zoomedplot = 0.01
markersize = 10
ax[0].scatter(combo['Units_Best'],combo['Units_CensusBest'], alpha = alpha_zoomedplot, s = markersize)
ax[1].scatter(combo['Units_Best'],combo['Average'], alpha = alpha_zoomedplot, s = markersize)
ax[2].scatter(combo['Units_Best'],combo['NSI_MinResUnits'], alpha = alpha_zoomedplot, s = markersize)
ax[3].scatter(combo['Units_Best'],combo['NSI_MaxResUnits'], alpha = alpha_zoomedplot, s = markersize)

ax[0].set_title('Census Unit Estimate')
ax[1].set_title('Average of NSI Range')
ax[2].set_title('Minimum of NSI Range')
ax[3].set_title('Maximum of NSI Range')

ax[0].set_yscale('log')
ax[0].set_xscale('log')

# Add correlation
titles = [f"Corr. Coeff = {round(np.corrcoef(combo['Units_Best'], combo['Units_CensusBest'])[0,1], 2)}",
          f"Corr. Coeff = {round(np.corrcoef(combo['Units_Best'], combo['Average'])[0,1], 2)}",
          f"Corr. Coeff = {round(np.corrcoef(combo['Units_Best'], combo['NSI_MinResUnits'])[0,1], 2)}",
          f"Corr. Coeff = {round(np.corrcoef(combo['Units_Best'], combo['NSI_MaxResUnits'])[0,1], 2)}"]
for i in range(4):
    ax[i].text(0.05, 0.95, titles[i], fontsize=10, ha='left', va='top', transform=ax[i].transAxes)

titles = [f"Number of RES3F = {len(national_res[national_res['Units_CensusEstimate_OC'].str.contains('RES3F')])}",
          f"Number of RES3F = {len(national_res[national_res['Average_OC'].str.contains('RES3F')])}",
          f"Number of RES3F = {len(national_res[national_res['NSI_MinResUnits_OC'].str.contains('RES3F')])}",
          f"Number of RES3F = {len(national_res[national_res['NSI_MaxResUnits_OC'].str.contains('RES3F')])}"]

for i in range(4):
    ax[i].text(0.05, 0.9, titles[i], fontsize=10, ha='left', va='top', transform=ax[i].transAxes)

plt.xlim([0.5,100])
plt.ylim([0.5,100])

plt.tight_layout()
plt.savefig(fig_dir + "census_correlation.svg", format="svg",  bbox_inches='tight', pad_inches=0)
plt.savefig(fig_dir + "census_correlation.png", bbox_inches='tight', pad_inches=0)
# plt.close()

# Print number of RES3F in best estimate inventory 
print(len(best[best['OccupancyClass_Best'].str.contains('RES3F')]), 'RES3F in best estimate inventory')